# Pipeline de atualização – Ponto de Controle

- Conecta aos sheets de **origem** e **destino**
- Gera / filtra / normaliza dados
- Escreve apenas linhas novas (dry-run opcional)

**Como executar**

```bash
poetry install
poetry run python ponto_de_controle_notebook.py --dry-run   # só loga
poetry run python ponto_de_controle_notebook.py             # grava no sheet
```

---
_Cada célula imprime o estado dos DataFrames para facilitar o rastreio._


Imports e constantes – todos num único bloco para facilitar lint/format.

In [ ]:
from __future__ import annotations

import argparse
import logging
import os
from collections import OrderedDict
from datetime import date
from pathlib import Path
from typing import Iterable

import pandas as pd
from dotenv import load_dotenv

from transform.extract import read_df 
from transform.transform.utils.campos_calculados import (
    DEFAULT_DEST_COLUMNS,
    add_key_creative,
    dedupe_by_key_creative,
    make_id_ponto_de_controle,
)
from transform.transform.utils.datas import concat_period, normalize_date_to_str_DD_M_YYYY
from transform.transform.utils.normalize import normalize_vehicle
from transform.transform.utils.write_dataframe_to_sheet import write_dataframe_to_sheet

logger = logging.getLogger(__name__)
pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", None)

In [ ]:
# ---------------------------------------------------------------------
# Config – pode ser sobrescrita por variáveis de ambiente (.env)
# ---------------------------------------------------------------------
load_dotenv()  # carrega .env local, se existir

ORIGIN_SHEET_ID: str = os.getenv("ORIGIN_SHEET_ID", "")
ORIGIN_TAB: str = os.getenv("ORIGIN_TAB", "modeloGeral")

DEST_SHEET_ID: str = os.getenv("DEST_SHEET_ID", "")
DEST_TAB: str = os.getenv("DEST_TAB", "IMPULSIONAMENTOS 2025")
HEAD_ROW_DEST: int = int(os.getenv("HEAD_ROW_DEST", "4"))  # zero-based

GOOGLE_CREDS_PATH: Path = Path(os.getenv("GOOGLE_CREDS_PATH", "creds.json"))

MIN_DATE = date(2025, 6, 1)
DEST_COLUMNS: list[str] = list(OrderedDict.fromkeys(DEFAULT_DEST_COLUMNS))  # garante unicidade
assert len(DEST_COLUMNS) == 11, "DEST_COLUMNS deve conter 11 rótulos únicos"

print("▶ DEST_COLUMNS:", DEST_COLUMNS)

## Auxiliares

In [ ]:
def load_google_creds() -> str:
    """Retorna o JSON de credenciais para uso na API Google."""
    return GOOGLE_CREDS_PATH.read_text(encoding="utf-8")


def debug_shape(df: pd.DataFrame, *, name: str) -> None:
    """Imprime forma, colunas e as 5 primeiras linhas de `df`."""
    print(f"▼ {name}: {df.shape[0]} × {df.shape[1]}")
    display(df.head())

## 1 · Extrair & preparar **origem**

In [ ]:
def read_origin_df() -> pd.DataFrame:
    """
    Lê planilha de origem, gera `key_creative`, filtra por data mínima
    e garante unicidade.
    """
    logger.info("Lendo origem %s › %s …", ORIGIN_SHEET_ID, ORIGIN_TAB)
    df = read_df(sheet_id=ORIGIN_SHEET_ID, tab=ORIGIN_TAB, header_row=0)

    # key_creative + dedup
    df = add_key_creative(df)
    df = dedupe_by_key_creative(df)

    # filtro temporal
    df["date_dt"] = pd.to_datetime(df["date"], errors="coerce").dt.date
    df = df[df["date_dt"] >= MIN_DATE].copy()
    df.drop(columns=["date_dt"], inplace=True)

    debug_shape(df, name="df_origin (pos-filtro)")
    assert df["key_creative"].ne("").all(), "Há key_creative vazio!"
    return df


df_origin = read_origin_df()  # executa já nesta célula

## 2 · Transformar para colunas de destino

In [ ]:
def transform_df(df: pd.DataFrame) -> pd.DataFrame:
    """Converte `df` para o layout de destino e calcula `__ID__`."""
    df2 = df.copy()

    df2["Data"] = df2["start"].apply(normalize_date_to_str_DD_M_YYYY)
    df2["Periodo"] = df2.apply(lambda r: concat_period(r["start"], r["end"]), axis=1)
    df2["Veiculo"] = df2["Veiculo"].apply(normalize_vehicle)

    df2["Link conteúdos impulsionados"] = df2.get("URL_do_Anuncio", "")
    df2["Agência"] = "De Brito"
    df2["Editoria"] = df2["Campanha"]
    df2["Objetivo"] = df2.get("objective", "")
    df2[["Meta", "Status", "Resultado"]] = ""

    df_t = df2.reindex(columns=DEST_COLUMNS, fill_value="")
    df_t["__ID__"] = df_t.apply(
        make_id_ponto_de_controle, axis=1, columns=DEST_COLUMNS
    )
    debug_shape(df_t, name="df_transf")
    return df_t


df_transf = transform_df(df_origin)

## 3 · Extrair **destino** e deduplicar cabeçalho

In [ ]:
# ⬇️  Coloque esta célula (ou substitua a anterior) no notebook
def read_destination_df() -> pd.DataFrame:
    """
    Lê a planilha‐destino, resolve colunas duplicadas, normaliza em DEST_COLUMNS
    e devolve um DataFrame único por ``__ID__``.

    • Se houver cabeçalhos repetidos (“Data”, “Data.1”…), renomeia de forma
      determinística usando um algoritmo próprio (não depende de APIs internas
      do pandas, portanto funciona em qualquer versão).
    • Sempre reindexa com DEST_COLUMNS → colunas faltantes recebem “”.
    • Gera/valida ``__ID__`` e elimina registros duplicados.
    """
    logger.info("Lendo destino %s › %s …", DEST_SHEET_ID, DEST_TAB)
    df = read_df(sheet_id=DEST_SHEET_ID, tab=DEST_TAB, header_row=HEAD_ROW_DEST)

    # ────────────────────────────────────────────────────────────────────
    # 1) Cabeçalhos duplicados  → Data, Data.1, Data.2 …
    # -------------------------------------------------------------------
    if df.columns.duplicated().any():
        logger.warning("Cabeçalhos duplicados detectados – renomeando")

        def _dedup_cols(cols: Iterable[str]) -> list[str]:
            """Gera nomes únicos preservando ordem:  'A', 'A' → 'A', 'A.1', …"""
            seen: dict[str, int] = {}
            new_cols: list[str] = []
            for col in cols:
                k = seen.get(col, 0)
                new_cols.append(col if k == 0 else f"{col}.{k}")
                seen[col] = k + 1
            return new_cols

        df.columns = _dedup_cols(df.columns)

    # ────────────────────────────────────────────────────────────────────
    # 2) Reindexa / normaliza
    # -------------------------------------------------------------------
    df = df.reindex(columns=DEST_COLUMNS, fill_value="")

    # ────────────────────────────────────────────────────────────────────
    # 3) Gera __ID__ e deduplica linhas
    # -------------------------------------------------------------------
    df["__ID__"] = df.apply(make_id_ponto_de_controle, axis=1, columns=DEST_COLUMNS)
    before = len(df)
    df = df.drop_duplicates("__ID__", keep="first").reset_index(drop=True)
    logger.info("Destino: %d → %d linhas únicas", before, len(df))

    # ────────────────────────────────────────────────────────────────────
    # 4) Debug helpers
    # -------------------------------------------------------------------
    
    debug_shape(df, name="df_dest")
    assert not df["__ID__"].duplicated().any(), "__ID__ duplicado no destino"

    return df


## 4 · Diferença & escrita

In [ ]:
# 4 · Diferença & escrita

from typing import Tuple

def diff_new_rows(src: pd.DataFrame, dst: pd.DataFrame) -> pd.DataFrame:
    """
    Retorna registros de `src` cujo `__ID__` não existe em `dst`.
    1) Verifica pré-condições
    2) Filtra novos registros
    3) Exibe debug (contagens + head)
    4) Validações finais
    """
    # 1) Pré-condições
    assert "__ID__" in src.columns, "Coluna '__ID__' ausente em src"
    assert "__ID__" in dst.columns, "Coluna '__ID__' ausente em dst"

    # 2) Filtrar novos registros
    mask_new = ~src["__ID__"].isin(dst["__ID__"])
    df_new = src.loc[mask_new].copy()

    # 3) Debug e inspeção rápida
    total_src = len(src)
    total_dst = len(dst)
    total_new = len(df_new)
    print(f"▶ Src: {total_src} linhas · Dst: {total_dst} linhas")
    print(f"▶ Novas linhas identificadas: {total_new}")
    display(df_new.head())

    # 4) Validações finais
    assert total_new <= total_src, "Número de novos registros maior que o total de src"
    assert not df_new["__ID__"].duplicated().any(), "__ID__ duplicado em df_new"

    return df_new

# 4.1 • Extrair destino
df_dest = read_destination_df()
assert "__ID__" in df_dest.columns, "Coluna '__ID__' ausente em df_dest"

# 4.2 • Executar diff
df_new = diff_new_rows(df_transf, df_dest)

# 4.3 • Debug final antes de escrever
print(f"▶ Preparado para gravar {len(df_new)} novas linhas (de {len(df_transf)} totais).")
debug_shape(df_new, name="df_new (a gravar)")


In [ ]:
def write_df_to_sheet_final(df_new: pd.DataFrame, *, dry_run: bool) -> None:
    """
    Grava `df_new` no destino se `dry_run` for False.
    Linha inicial = HEAD_ROW_DEST + dados existentes + 1.
    """
    if dry_run:
        logger.info("DRY-RUN: %d linhas seriam gravadas", len(df_new))
        return
    if df_new.empty:
        logger.info("Nada a gravar – destino já está atualizado.")
        return

    creds_json = load_google_creds()
    start_row = HEAD_ROW_DEST + 1 + len(df_dest) + 1  # header + dados + linha vazia
    write_dataframe_to_sheet(
        spreadsheet_id=DEST_SHEET_ID,
        sheet_name=DEST_TAB,
        df=df_new.drop(columns="__ID__", errors="ignore"),
        start_row=start_row,
        include_header=False,
        google_credentials_json=creds_json,
    )
    logger.info("Gravadas %d linhas na linha %d", len(df_new), start_row)

## 5 · Função `main` + CLI

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# 5 · Função main + CLI
# ────────────────────────────────────────────────────────────────────────────
def main(*, dry_run: bool) -> None:
    """
    Orquestra todo o pipeline:
      1) Extrai origem e destino (já executados no notebook)
      2) Identifica novas linhas em df_new
      3) Exibe o DataFrame que será gravado (sem a coluna __ID__)
      4) Grava (ou simula) no Google Sheets
    """
    logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
    logger.info("── Início do pipeline ──")

    total_new = len(df_new)
    logger.info("▶ Total de linhas a gravar: %d", total_new)

    if total_new:
        # em notebook, display; em script, cai no except e printa
        try:
            from IPython.display import display  # type: ignore
            display(df_new.drop(columns="__ID__", errors="ignore"))
        except ImportError:
            print(df_new.drop(columns="__ID__", errors="ignore"))

    # chama a escrita — dentro dela a coluna __ID__ é removida de qualquer forma
    write_df_to_sheet_final(df_new, dry_run=dry_run)

    logger.info("── Fim ──")


if __name__ == "__main__" and "get_ipython" not in globals():
    parser = argparse.ArgumentParser(description="Atualiza ponto de controle")
    parser.add_argument("--dry-run", action="store_true", help="não grava no sheet")
    args = parser.parse_args()
    main(dry_run=args.dry_run)
else:
    # Em notebook, executamos em dry-run para segurança
    main(dry_run=True)
